# Client Segmentation Clustering
This notebook clusters clients by behavior and compares clustering algorithms.

Le **nettoyage** (cohérence des clés, valeurs manquantes post-fusion, valeurs infinies) est détaillé **dans ce notebook** avant le `ColumnTransformer` et le clustering.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if (ROOT / 'ml-workspace').exists():
    ROOT = ROOT / 'ml-workspace'
CLIENTS_PATH = ROOT / 'data/raw/clients.csv'
PROJECTS_PATH = ROOT / 'data/raw/projects.csv'
RESULTS_PLOTS = ROOT / 'results/plots'
RESULTS_METRICS = ROOT / 'results/metrics'
RESULTS_PLOTS.mkdir(parents=True, exist_ok=True)
RESULTS_METRICS.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
clients = pd.read_csv(CLIENTS_PATH)
projects = pd.read_csv(PROJECTS_PATH)
print('clients:', clients.shape, '| projects:', projects.shape)
display(clients.head(3))
display(projects.head(3))

## Construction du jeu client + agrégats projets
Agrégation par `client_id` puis jointure ; les clients sans projet conservent des NaN sur les colonnes agrégées.

In [ ]:
agg = projects.groupby('client_id', as_index=False).agg(
    mean_completion_ratio=('completion_ratio', 'mean'),
    mean_delay_days=('avg_task_delay_days', 'mean'),
    project_success_rate=('success_flag', 'mean'),
    mean_budget_usd=('budget_usd', 'mean'),
    avg_complexity=('complexity_score', 'mean'),
    total_projects=('project_id', 'count')
)
merged = clients.merge(agg, on='client_id', how='left')
print('Lignes après merge:', len(merged))
dup_ids = merged['client_id'].duplicated().sum()
print('client_id dupliqués:', int(dup_ids))
display(merged.isna().mean().sort_values(ascending=False).head(15))

## Nettoyage des données (étapes explicites)
1. Remplacement des `inf` / `-inf` issus d'agrégations éventuelles par NaN.
2. Imputation des colonnes numériques post-merge par la **médiane** (robuste).
3. Imputation des colonnes catégorielles par le **mode** (ou libellé `'unknown'`).
4. Le pipeline ci-dessous refait une imputation fit sur les données pour alignement avec `run_all_models.py`.

In [ ]:
df = merged.copy()
df = df.replace([np.inf, -np.inf], np.nan)
null_before = df.isna().sum()
num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols_all:
    if df[c].isna().any():
        med = df[c].median()
        df[c] = df[c].fillna(med)
cat_cols_all = df.select_dtypes(exclude=[np.number]).columns.tolist()
if 'client_id' in cat_cols_all:
    cat_cols_all.remove('client_id')
for c in cat_cols_all:
    if df[c].isna().any():
        mode = df[c].mode(dropna=True)
        fill = mode.iloc[0] if len(mode) else 'unknown'
        df[c] = df[c].fillna(fill)
null_after = df.isna().sum()
print('Total NaN restants:', int(null_after.sum()))
display(pd.DataFrame({'avant': null_before, 'apres': null_after}).fillna(0).astype(int).T)
display(df.head())

## Pré-traitement (encodage + mise à l'échelle) et PCA pour la visualisation
`fit_transform` sur tout `X` ici pour le clustering exploratoire ; en production on isolerait un jeu d'apprentissage.

In [ ]:
X = df.drop(columns=['client_id'])
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('standardize', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])
Xp = preprocessor.fit_transform(X)
Xp_dense = Xp.toarray() if hasattr(Xp, 'toarray') else Xp
pca = PCA(n_components=2, random_state=SEED)
X2 = pca.fit_transform(Xp_dense)

In [ ]:
algorithms = {
    'kmeans': KMeans(n_clusters=4, random_state=SEED, n_init=10),
    'agglomerative': AgglomerativeClustering(n_clusters=4),
    'dbscan': DBSCAN(eps=1.25, min_samples=7)
}
rows = []
for name, algo in algorithms.items():
    labels = algo.fit_predict(Xp_dense)
    valid = labels != -1
    unique = np.unique(labels[valid]) if valid.any() else np.array([])
    if len(unique) > 1 and valid.sum() > len(unique):
        sil = silhouette_score(Xp_dense[valid], labels[valid])
        ch = calinski_harabasz_score(Xp_dense[valid], labels[valid])
        db = davies_bouldin_score(Xp_dense[valid], labels[valid])
    else:
        sil, ch, db = np.nan, np.nan, np.nan

    rows.append({
        'algorithm': name,
        'num_clusters': len(np.unique(labels[labels != -1])),
        'noise_ratio': float((labels == -1).mean()),
        'silhouette': sil,
        'calinski_harabasz': ch,
        'davies_bouldin': db
    })

    plt.figure(figsize=(7,5))
    sns.scatterplot(x=X2[:,0], y=X2[:,1], hue=labels, palette='tab10', s=42, legend=False)
    plt.title(f'Cluster Projection - {name}')
    plt.xlabel('PCA 1')
    plt.ylabel('PCA 2')
    plt.tight_layout()
    plt.savefig(RESULTS_PLOTS / f'clustering_projection_{name}.png', dpi=160)
    plt.show()

metrics_df = pd.DataFrame(rows).sort_values('silhouette', ascending=False, na_position='last')
display(metrics_df)
metrics_df.to_csv(RESULTS_METRICS / 'clustering_metrics.csv', index=False)
metrics_df.to_json(RESULTS_METRICS / 'clustering_metrics.json', orient='records', indent=2)

In [ ]:
ks = range(2, 9)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(Xp_dense)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(Xp_dense, labels))

fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].plot(list(ks), inertias, marker='o')
axes[0].set_title('KMeans Elbow Curve')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[1].plot(list(ks), silhouettes, marker='o')
axes[1].set_title('KMeans Silhouette Trend')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')
plt.tight_layout()
plt.savefig(RESULTS_PLOTS / 'clustering_kmeans_curves.png', dpi=160)
plt.show()
print('Best clustering algorithm (highest silhouette):', metrics_df.iloc[0]['algorithm'])